# AI Tutor — offline-model eval on Google Colab (free T4)

Evaluate bigger open-source tutor models than an 8 GB laptop can run, using the
**same harness** (tutor = OSS via Ollama; judge + student-sim = Anthropic).

**Before you start**
1. Runtime → **Change runtime type → T4 GPU**.
2. Add these **Colab Secrets** (🔑 icon in the left sidebar), each toggled
   *Notebook access ON*:
   - `GH_TOKEN` — a GitHub **classic** Personal Access Token with the **`repo`**
     scope. Make it at github.com/settings/tokens → *Generate new token
     (classic)* → check **repo**. A classic token works on `eai6/ai-tutor` because you
     are a **collaborator** (a fine-grained token would only work if you *owned*
     the repo).
   - `ANTHROPIC_API_KEY` — required (judge + student-simulator).
   - `GOOGLE_API_KEY` and `OPENAI_API_KEY` — keep these too so the judge/grader
     cross-vendor cascade matches the laptop runs (comparable scores).

**T4 fits models up to ~14B q4.** For the bigger A100/Colab-Pro tier (commented
out in Cell 8), use a Colab Pro A100 runtime — nothing else changes.

## Cell 1 — confirm GPU + mount Drive (Drive persists results across disconnects)

In [3]:
!nvidia-smi -L
from google.colab import drive
drive.mount('/content/drive')

GPU 0: Tesla T4 (UUID: GPU-047fa8f3-7095-2807-7f2c-8fdd7e11d18c)
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## Cell 2 — clone the repo (branch `pixeldesignlabs-dev-portuguese`) using the GH_TOKEN classic PAT

In [4]:
from google.colab import userdata
import subprocess, os
tok = (userdata.get('GH_TOKEN') or '').strip()   # strip stray spaces/newlines
assert tok and ' ' not in tok, "GH_TOKEN missing or contains a space — re-save the secret with no whitespace"
url = f"https://{tok}@github.com/eai6/ai-tutor.git"
subprocess.run(['rm', '-rf', '/content/ai-tutor'], check=True)
subprocess.run(['git', 'clone', '--depth', '1', '-b', 'pixeldesignlabs-dev-portuguese', url, '/content/ai-tutor'], check=True)
os.chdir('/content/ai-tutor')
print('cloned at', os.getcwd())

cloned at /content/ai-tutor


## Cell 3 — fix hardcoded laptop paths (essential)

In [5]:
!sed -i 's#/home/daniel/Documents/work/Nyansapo/web/ai-tutor#/content/ai-tutor#g; s#\$ROOT/venv/bin/python#python#g; s#venv/bin/python#python#g' offline_eval/*.py offline_eval/*.sh

## Cell 4 — install deps + start Ollama (a few min; ignore pip resolver warnings)

In [7]:
!pip install -q -r requirements.txt
# The Ollama installer is now zstd-compressed; the Colab VM lacks zstd, so install
# it first (otherwise the installer aborts and `ollama` is never created).
!apt-get -qq install -y zstd || (apt-get -qq update && apt-get -qq install -y zstd)
!curl -fsSL https://ollama.com/install.sh | sh
import subprocess, time, shutil
assert shutil.which('ollama'), "ollama did not install — check the install output above (zstd?)"
subprocess.Popen(['ollama', 'serve'],
                 stdout=open('/content/ollama.log', 'w'),
                 stderr=subprocess.STDOUT)
for _ in range(30):
    if subprocess.run(['bash', '-c', 'ollama list'], capture_output=True).returncode == 0:
        print('ollama ready'); break
    time.sleep(2)
else:
    print('ollama NOT ready — check /content/ollama.log')

Selecting previously unselected package zstd.
(Reading database ... 122403 files and directories currently installed.)
Preparing to unpack .../zstd_1.4.8+dfsg-3build1_amd64.deb ...
Unpacking zstd (1.4.8+dfsg-3build1) ...
Setting up zstd (1.4.8+dfsg-3build1) ...
Processing triggers for man-db (2.10.2-1) ...
>>> Cleaning up old version at /usr/local/lib/ollama
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.
ollama ready


## Cell 5 — **required** — write .env from Colab Secrets
`.env` isn't in the repo (gitignored). Keep **all three** keys so the judge/grader cascade matches the laptop runs (comparable scores).

In [8]:
from google.colab import userdata
open('.env', 'w').write(
    "SECRET_KEY=colab-eval\nDEBUG=True\nEMBEDDING_BACKEND=sqlite\n"
    f"ANTHROPIC_API_KEY={userdata.get('ANTHROPIC_API_KEY')}\n"
    f"GOOGLE_API_KEY={userdata.get('GOOGLE_API_KEY')}\n"
    f"OPENAI_API_KEY={userdata.get('OPENAI_API_KEY')}\n")
print('.env written')

.env written


## Cell 6 — fresh DB + eval fixtures

In [9]:
!python manage.py migrate
!python manage.py loaddata evals/fixtures/institution.json evals/fixtures/lessons.json

Operations to perform:
  Apply all migrations: accounts, admin, auth, benchmark, contenttypes, curriculum, dashboard, llm, media_library, safety, sessions, support, token_blacklist, tutoring
Running migrations:
  Applying contenttypes.0001_initial... OK
  Applying auth.0001_initial... OK
  Applying accounts.0001_initial... OK
  Applying llm.0001_initial... OK
  Applying llm.0002_promptpack_content_generation_prompt_and_more... OK
  Applying accounts.0002_studentprofile... OK
  Applying accounts.0003_staffinvitation... OK
  Applying accounts.0004_alter_membership_role_alter_staffinvitation_role... OK
  Applying accounts.0005_institution_accent_color_institution_custom_css_and_more... OK
  Applying accounts.0006_platformconfig_remove_institution_custom_css... OK
  Applying accounts.0007_alter_membership_role_alter_staffinvitation_email_and_more... OK
  Applying accounts.0008_move_branding_to_platformconfig... OK
  Applying llm.0003_make_promptpack_institution_nullable... OK
  Applying ac

## Cell 7 — persist results to Drive (seed with the committed laptop results, then symlink)
This makes resume survive disconnects AND gives a **combined** leaderboard (laptop small models + the big models you run here).

In [10]:
!mkdir -p /content/drive/MyDrive/ai-tutor-eval-results
# seed the Drive folder with the laptop results committed in the repo (no-clobber)
!cp -n offline_eval/results/*.json /content/drive/MyDrive/ai-tutor-eval-results/ 2>/dev/null || true
!rm -rf offline_eval/results && ln -s /content/drive/MyDrive/ai-tutor-eval-results offline_eval/results
!ls offline_eval/results/

gemma2_2b.json		  llama3.2_1b.json		qwen2.5_0.5b.json
granite3.1-dense_2b.json  llama3.2_3b.json		qwen2.5_1.5b.json
granite3.1-moe_3b.json	  llama3-groq-tool-use_8b.json	qwen2.5_3b.json
hermes3_3b.json		  nemotron-mini.json


## Cell 8 — choose the big-model matrix + seed configs
All models below are **tool-calling capable** (the engine requires it). The **T4 tier** (≤~14B q4) runs on free Colab. The **A100 / Colab-Pro tier** (>16GB VRAM) is commented out so it won't OOM a T4 — uncomment those lines only on an A100 runtime. Trim the list to control runtime (~20–40 min each).

In [11]:
open('offline_eval/models.txt', 'w').write('''\
# ============ T4 (free Colab, 16GB) tier — fits ~14B q4 ============
qwen2.5:7b            big
llama3.1:8b           big
mistral:7b            big
glm4:9b               big
qwen2.5:14b           big
phi4                  big
mistral-nemo:12b      big
granite3.1-dense:8b   big
hermes3:8b            big
aya-expanse:8b        big    # Cohere — multilingual, relevant for MZ/TZ
falcon3:10b           big
command-r7b           big    # Cohere 7B — multilingual + tools

# ============ A100 / Colab-Pro tier — needs >16GB VRAM; UNCOMMENT on A100 ===
# mistral-small:24b   xl
# qwen2.5:32b         xl
# command-r:35b       xl     # strong tool-use + multilingual
# mixtral:8x7b        xl     # 47B MoE
# llama3.3:70b        xl
# qwen2.5:72b         xl
# athene-v2:72b       xl
# command-r-plus:104b xl     # 104B — needs an 80GB A100
''')
!python offline_eval/seed_ollama_configs.py

  created: local_ollama/qwen2.5:7b
  created: local_ollama/llama3.1:8b
  created: local_ollama/mistral:7b
  created: local_ollama/glm4:9b
  created: local_ollama/qwen2.5:14b
  created: local_ollama/phi4
  created: local_ollama/mistral-nemo:12b
  created: local_ollama/granite3.1-dense:8b
  created: local_ollama/hermes3:8b
  created: local_ollama/aya-expanse:8b
  created: local_ollama/falcon3:10b
  created: local_ollama/command-r7b

Seeded 12 ollama configs (12 created, 0 updated).


In [14]:
import os, glob, subprocess
scored = {os.path.basename(p)[:-5] for p in glob.glob('offline_eval/results/*.json')}
out = subprocess.run(['ollama','list'], capture_output=True, text=True).stdout
for line in out.splitlines()[1:]:
    if not line.strip(): continue
    tag = line.split()[0]
    if tag.replace(':','_').replace('/','_') in scored:
        print('removing scored:', tag); subprocess.run(['ollama','rm',tag])
subprocess.run(['ollama','rm','qwen2.5:0.5b'])
subprocess.run('pip cache purge; apt-get clean; df -h / | tail -1', shell=True)

CompletedProcess(args='pip cache purge; apt-get clean; df -h / | tail -1', returncode=0)

In [16]:
!df -h / | tail -1

overlay         113G  100G   13G  89% /


In [15]:
p = 'offline_eval/run_matrix.sh'; s = open(p).read()
if 'ollama rm' not in s:
    s = s.replace('ollama stop "$tag" >/dev/null 2>&1 || true',
        'ollama stop "$tag" >/dev/null 2>&1 || true\n'
        '  [ -f "$RESULTS/${safe}.json" ] && ollama rm "$tag" >/dev/null 2>&1 || true')
    open(p,'w').write(s); print('patched — weights deleted after each score')
else:
    print('already patched')

patched — weights deleted after each score


In [19]:

import subprocess, time
subprocess.Popen(['ollama','serve'],
                 stdout=open('/content/ollama.log','w'), stderr=subprocess.STDOUT)
for _ in range(30):
    if subprocess.run(['bash','-c','ollama list'], capture_output=True).returncode == 0:
        print('ollama ready'); break
    time.sleep(2)
else:
    print('NOT ready — check /content/ollama.log')

ollama ready


## Cell 9 — run the sweep (pulls + scores each model; resume-safe; ~20–40 min/model on T4)

In [20]:
!SIMPLE_TUTOR_ENGINE=1 CLEANUP_MODELS=1 bash offline_eval/run_matrix.sh

>> Seeding local_ollama ModelConfig rows...
  updated: local_ollama/qwen2.5:7b
  updated: local_ollama/llama3.1:8b
  updated: local_ollama/mistral:7b
  updated: local_ollama/glm4:9b
  updated: local_ollama/qwen2.5:14b
  updated: local_ollama/phi4
  updated: local_ollama/mistral-nemo:12b
  updated: local_ollama/granite3.1-dense:8b
  updated: local_ollama/hermes3:8b
  updated: local_ollama/aya-expanse:8b
  updated: local_ollama/falcon3:10b
  updated: local_ollama/command-r7b

Seeded 12 ollama configs (0 created, 12 updated).
>> Engine: SIMPLE_TUTOR_ENGINE=1   Mode: --single-turn
>> Model weights: /content/ai-tutor/offline_eval/ollama_models

==================== qwen2.5:7b (big) — already done, skipping ====================

==================== llama3.1:8b (big) — already done, skipping ====================

==================== mistral:7b (big) — already done, skipping ====================

==================== glm4:9b (big) — already done, skipping ====================

==============

## Cell 10 — combined leaderboard (run anytime)

In [21]:
!python offline_eval/aggregate.py


MODEL                       PASS    RATE  RUBRIC  ERR  BOTTLENECK
------------------------------------------------------------------------------
qwen2.5_14b                33/60     55%    0.66    0  math (11)
mistral-nemo_12b           32/60     53%    0.67    0  math (10)
qwen2.5_7b                 31/60     52%    0.71    0  crosscutting (9)
qwen2.5_3b                 27/60     45%    0.61    0  persona_handling (11)
glm4_9b                    26/60     43%    0.60    0  crosscutting (9)
granite3.1-dense_8b        20/60     33%    0.56    0  math (13)
llama3.1_8b                20/60     33%    0.53    0  persona_handling (13)
mistral_7b                 19/60     32%    0.54    0  math (15)
llama3.2_3b                17/60     28%    0.46    0  persona_handling (13)
qwen2.5_1.5b               17/60     28%    0.50    0  persona_handling (13)
hermes3_8b                 13/60     22%    0.43    0  math (15)
llama3-groq-tool-use_8b     13/60     22%    0.46    0  math (13)
command-r7b

## After a Colab disconnect (free tier: ~90 min idle / ~12 h max)
Re-run **Cells 1–8**, then **Cell 9** again. Because results live on Drive (Cell 7),
`run_matrix.sh` **skips already-scored models** and continues.

To pick up new commits on the branch, just re-run **Cell 2** (it re-clones).

**Tips:** keep the tab active (free Colab kills idle sessions); a model interrupted
mid-run restarts (resume only skips *completed* models); each model is also bound on
the Anthropic judge calls, so plan 1–3 models per session.

To pull these results back to your laptop: copy the JSONs from
`MyDrive/ai-tutor-eval-results/` into the repo's `offline_eval/results/` and run
`python offline_eval/aggregate.py`.

**If GLM under-scores:** it leaks tool calls as text under the full prompt. Set
`OLLAMA_DEBUG_RAW=1` before Cell 9, then check `offline_eval/results/glm4_9b.log`
for an `[OllamaToolLeak]` line and share it — the parser can then be extended to
match glm4's exact format.